In [1]:
!pip install -q streamlit langchain langchain-google-genai langchain-community faiss-cpu
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙

In [2]:
import os

# Create data directory
os.makedirs("data", exist_ok=True)

# 1. Create sample email data for RAG
email_examples = """Subject: Follow-up on Proposal
Tone: Professional
Dear [Name],
I hope this email finds you well. I am following up on our previous discussion regarding the project proposal. Please let me know if you have any questions or require additional details.
Best regards,
[Sender]

Subject: Quick Question
Tone: Friendly
Hi [Name],
Hope you're having a great week! Just wanted to quickly check in regarding the update. Let me know when you have a moment to chat.
Cheers,
[Sender]
"""

with open("data/email_examples.txt", "w") as f:
    f.write(email_examples)

# 2. Create requirements.txt
requirements = """streamlit
langchain
langchain-google-genai
langchain-community
faiss-cpu
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

# 3. Create app.py file
app_code = '''import streamlit as st
import os
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

st.set_page_config(page_title="AI Email Assistant", page_icon="✉️", layout="wide")

st.markdown("<h1 style='text-align: center; color: #1E88E5;'>✉️ Smart Email Writing Assistant (RAG Enabled)</h1>", unsafe_allow_html=True)

api_key = st.sidebar.text_input("Enter Google Gemini API Key", type="password")

if not api_key:
    st.info("Please enter your Google Gemini API Key in the sidebar to start.")
    st.stop()

@st.cache_resource
def init_rag(key):
    loader = TextLoader("data/email_examples.txt")
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    docs = text_splitter.split_documents(documents)
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=key)
    return FAISS.from_documents(docs, embeddings)

try:
    vectorstore = init_rag(api_key)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
except Exception as e:
    st.error(f"Error loading RAG: {e}")
    st.stop()

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", google_api_key=api_key, temperature=0.3)

tab1, tab2 = st.tabs(["🚀 Generate Email", "🛠️ Refine & Edit"])

with tab1:
    col1, col2 = st.columns([1, 1])
    with col1:
        recipient = st.text_input("Recipient", placeholder="e.g., Hiring Manager")
        purpose = st.text_area("Purpose / Key Points", placeholder="e.g., Ask for budget approval")
        tone = st.selectbox("Tone", ["Professional", "Friendly", "Formal", "Persuasive"])
        btn = st.button("Generate Email")

    with col2:
        if btn and purpose and recipient:
            with st.spinner("Generating..."):
                docs = retriever.get_relevant_documents(f"Tone: {tone} Purpose: {purpose}")
                context = "\\n".join([d.page_content for d in docs])
                template = """
                Context:
                {context}

                Write an email to {recipient} with tone '{tone}'.
                Purpose: {purpose}
                """
                prompt = PromptTemplate(template=template, input_variables=["context", "recipient", "tone", "purpose"])
                chain = LLMChain(llm=llm, prompt=prompt)
                res = chain.run(context=context, recipient=recipient, tone=tone, purpose=purpose)
                st.text_area("Result", value=res, height=250)

with tab2:
    text = st.text_area("Paste Email to Refine")
    c1, c2, c3 = st.columns(3)
    if c1.button("Rewrite") and text:
        st.write(llm.predict(f"Rewrite cleanly:\\n\\n{text}"))
    if c2.button("Shorten") and text:
        st.write(llm.predict(f"Shorten this email:\\n\\n{text}"))
    if c3.button("Fix Grammar") and text:
        st.write(llm.predict(f"Fix grammar:\\n\\n{text}"))
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("Project files created successfully!")

Project files created successfully!


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦

2026-08-10 15:51:33.687 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.10.47.222:8501

your url is: https://tiny-seals-train.loca.lt
